# Cacao flower microbiome (16S) — exploratory visualization

Feature table cleaned in R with two steps:
1. **Decontam** (`prevalence`, threshold = 0.55): 91 contaminant ASVs removed using extraction and negative PCR controls
2. **Off-target filter**: only `d__Bacteria`, excluding `o__Chloroplast` and `f__Mitochondria`

All samples are included (biological + controls) to show how controls cluster relative to biological samples.

**Sample ID encoding:**
| part | meaning |
|---|---|
| 1st letter `c` / `p` | flower type: **c**losed (insect-excluded) / **p**ollinator-visited (open) |
| letters 2-3 | unique farm ID (`ib`, `kk`, `mt`, …) |
| 1st digit | tree ID on that farm |
| 2nd digit | flower replicate (3 per tree) |
| suffix `EC` | extraction control |
| suffix `NC` | negative PCR control |
| suffix `PC` | mock community (positive control) |

**Visualizations in this notebook:**
1. Relative abundance heatmap (top-50 ASVs, log₁₀)
2. CLR-transformed abundance heatmap (same ASVs)
3. Bray-Curtis dissimilarity matrix
4. Aitchison distance matrix (Euclidean on CLR)

In [ ]:
import re
import colorsys

import biom
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.patches import Patch
from scipy.cluster.hierarchy import linkage
from scipy.spatial.distance import pdist, squareform
from skbio.stats.composition import clr

## Data loading

In [ ]:
DATA_DIR = "../data/real/cacao_flower_metabarcoding"

table_16s = biom.load_table(f"{DATA_DIR}/cfm_16s_bacteria_allsamples_feature-table.biom")
print(f"Loaded: {table_16s.shape[1]} samples × {table_16s.shape[0]} ASVs (bacteria only, decontaminated)")

# ── Parse sample metadata from IDs ───────────────────────────────────────────
CONTROL_LABELS = {"EC": "extraction_ctrl", "NC": "negative_ctrl", "PC": "positive_ctrl"}
FLOWER_TYPES   = {"c": "closed", "p": "open"}

def parse_sample_id(sid: str) -> dict:
    for suffix, label in CONTROL_LABELS.items():
        if sid.endswith(suffix):
            body = sid[: -len(suffix)]
            flower_letter = body[0] if body else ""
            farm_id = body[1:] if len(body) > 1 else ""
            return {"sample_id": sid, "sample_type": label,
                    "flower_type": FLOWER_TYPES.get(flower_letter, "unknown"),
                    "farm_id": farm_id, "tree_id": None}

    m = re.fullmatch(r"([cp])([a-z]{2})(\d)(\d)", sid)
    if m:
        flower_letter, farm_id, tree_digit, _ = m.groups()
        return {"sample_id": sid, "sample_type": "biological",
                "flower_type": FLOWER_TYPES.get(flower_letter, "unknown"),
                "farm_id": farm_id, "tree_id": f"{farm_id}{tree_digit}"}

    return {"sample_id": sid, "sample_type": "unknown",
            "flower_type": "unknown", "farm_id": "", "tree_id": None}

sample_ids = list(table_16s.ids(axis="sample"))
metadata   = pd.DataFrame([parse_sample_id(s) for s in sample_ids]).set_index("sample_id")

print(metadata["sample_type"].value_counts())
print()
metadata.head(8)

In [ ]:
# ── Build count + relative-abundance matrices ─────────────────────────────────
count_matrix = table_16s.matrix_data.toarray().T      # (n_samples, n_asvs)
asv_ids      = list(table_16s.ids(axis="observation"))

count_df     = pd.DataFrame(count_matrix, index=sample_ids, columns=asv_ids)
rel_abund_df = count_df.div(count_df.sum(axis=1), axis=0)

n_asvs_total = table_16s.shape[0]
print(f"count_df:     {count_df.shape}")
print(f"rel_abund_df: {rel_abund_df.shape}")

# ── Sample-type grouping ──────────────────────────────────────────────────────
def get_ids(meta, sample_type, flower_type=None):
    mask = meta["sample_type"] == sample_type
    if flower_type is not None:
        mask &= meta["flower_type"] == flower_type
    return meta.index[mask].tolist()

groups = {
    "closed":          get_ids(metadata, "biological", "closed"),
    "open":            get_ids(metadata, "biological", "open"),
    "extraction_ctrl": get_ids(metadata, "extraction_ctrl"),
    "negative_ctrl":   get_ids(metadata, "negative_ctrl"),
    "positive_ctrl":   get_ids(metadata, "positive_ctrl"),
}
sample_to_group = {sid: name for name, ids in groups.items() for sid in ids}

SAMPLE_TYPE_COLORS = {
    "closed":          "#4C72B0",
    "open":            "#DD8452",
    "extraction_ctrl": "#55A868",
    "negative_ctrl":   "#C44E52",
    "positive_ctrl":   "#8172B2",
}
CTRL_COLOR = "#cccccc"

# ── Farm + tree palettes ──────────────────────────────────────────────────────
farm_ids_uniq = sorted(f for f in metadata["farm_id"].unique() if f)
farm_palette  = dict(zip(farm_ids_uniq, sns.color_palette("Set2", len(farm_ids_uniq))))

def make_tree_palette(farm_ids_list, n_trees=7):
    pal = {}
    n_f = len(farm_ids_list)
    for i, farm in enumerate(farm_ids_list):
        hue = i / n_f
        for t in range(1, n_trees + 1):
            lightness = 0.75 - 0.45 * (t - 1) / (n_trees - 1)
            r, g, b = colorsys.hls_to_rgb(hue, lightness, 0.72)
            pal[f"{farm}{t}"] = (r, g, b)
    return pal

tree_palette_49 = make_tree_palette(farm_ids_uniq)

def farm_color(sid):
    return farm_palette.get(metadata.loc[sid, "farm_id"], CTRL_COLOR)

def tree_color(sid):
    tid = metadata.loc[sid, "tree_id"]
    if tid is None or (isinstance(tid, float) and np.isnan(tid)):
        return CTRL_COLOR
    return tree_palette_49.get(str(tid), CTRL_COLOR)

# ── Taxonomy: ASV → genus ─────────────────────────────────────────────────────
tax_df = pd.read_csv(f"{DATA_DIR}/cfm_16s_bacteria_allsamples_taxonomy.tsv", sep="\t", index_col=0)

def extract_genus(taxon):
    for part in str(taxon).split(";"):
        p = part.strip()
        if p.startswith("g__"):
            g = p[3:].strip()
            return g if g else "unclassified"
    return "unclassified"

asv_to_genus = tax_df["Taxon"].apply(extract_genus).to_dict()

# ── Top-50 ASVs + genus rename ────────────────────────────────────────────────
N_TOP    = 50
top_asvs = rel_abund_df.mean(axis=0).nlargest(N_TOP).index

genus_raw = [asv_to_genus.get(a, "unclassified") for a in top_asvs]
_cnt: dict = {}
final_rename = {}
for asv, g in zip(top_asvs, genus_raw):
    if genus_raw.count(g) > 1:
        _cnt[g] = _cnt.get(g, 0) + 1
        final_rename[asv] = f"{g} ({_cnt[g]})"
    else:
        final_rename[asv] = g

# ── Shared row colour strips ──────────────────────────────────────────────────
# Used in RA heatmap and CLR heatmap (rows = samples)
row_colors_samples = pd.DataFrame({
    "sample type": [SAMPLE_TYPE_COLORS[sample_to_group[s]] for s in sample_ids],
    "farm":        [farm_color(s) for s in sample_ids],
    "tree":        [tree_color(s) for s in sample_ids],
}, index=sample_ids)

# ── Shared legend handles ─────────────────────────────────────────────────────
legend_sample_type = [Patch(facecolor=c, label=n) for n, c in SAMPLE_TYPE_COLORS.items()]
legend_farm        = [Patch(facecolor=c, label=f) for f, c in farm_palette.items()]
legend_tree        = [
    Patch(facecolor=tree_palette_49.get(f"{farm}{t}", CTRL_COLOR), label=f"{farm}{t}")
    for farm in farm_ids_uniq for t in range(1, 8)
] + [Patch(facecolor=CTRL_COLOR, label="control")]

# Common kwargs for all three legend blocks
LEGEND_KW = dict(
    loc="upper center",
    bbox_transform=None,   # set per-figure below
    fontsize=9, title_fontsize=10,
    framealpha=0.9,
    borderaxespad=0,
    columnspacing=1.0,
    handlelength=1.2,
    handletextpad=0.5,
)

def add_legends(fig, y_stype=0.965, y_farm=0.930, y_tree=0.895):
    """Add the three stacked horizontal legends to a figure."""
    kw = dict(LEGEND_KW, bbox_transform=fig.transFigure)
    fig.legend(handles=legend_sample_type, title="sample type",
               bbox_to_anchor=(0.5, y_stype), ncol=len(SAMPLE_TYPE_COLORS), **kw)
    fig.legend(handles=legend_farm, title="farm",
               bbox_to_anchor=(0.5, y_farm), ncol=len(farm_palette), **kw)
    fig.legend(handles=legend_tree, title="tree (farm+#)",
               bbox_to_anchor=(0.5, y_tree), ncol=7, **kw)

print("Setup complete.")

## Relative abundance heatmap (top-50 ASVs)

Rows = samples, columns = top-50 ASVs by mean relative abundance across all samples.  
Values: log₁₀(RA + 1e-6). Clustering: Euclidean distance, average linkage.

In [ ]:
plot_log = np.log10(rel_abund_df[top_asvs] + 1e-6).rename(columns=final_rename)

# Row colour strips use sample ordering from plot_log.index (same as sample_ids)
row_colors_ra = row_colors_samples.loc[plot_log.index]

g_ra = sns.clustermap(
    plot_log,
    method="average",
    metric="euclidean",
    row_colors=row_colors_ra,
    figsize=(20, 24),
    yticklabels=False,
    xticklabels=True,
    cmap="YlOrRd",
    cbar_pos=None,
    dendrogram_ratio=(0.10, 0.15),
    colors_ratio=0.010,
)
g_ra.ax_heatmap.set_xticklabels(
    g_ra.ax_heatmap.get_xticklabels(), rotation=90, fontsize=9
)
g_ra.fig.subplots_adjust(top=0.72)

# Colorbar — manual for precise size control
cax_ra = g_ra.fig.add_axes([0.02, 0.755, 0.18, 0.006])
norm_ra = mpl.colors.Normalize(vmin=plot_log.values.min(), vmax=plot_log.values.max())
cb_ra   = g_ra.fig.colorbar(mpl.cm.ScalarMappable(norm=norm_ra, cmap="YlOrRd"),
                             cax=cax_ra, orientation="horizontal")
cb_ra.set_label("log₁₀(RA + 1e-6)", fontsize=10)
cb_ra.ax.tick_params(labelsize=9)

g_ra.fig.suptitle(
    f"Relative abundance — {len(sample_ids)} samples × top-{N_TOP} ASVs\n"
    f"(bacteria only, decontaminated; {n_asvs_total} ASVs total) — Euclidean, average linkage",
    x=0.5, y=0.995, ha="center", fontsize=13,
)
add_legends(g_ra.fig)
plt.show()

## Aitchison distance (CLR transformation)

Bray-Curtis is **not** compositionally coherent.  
The **Aitchison distance** is the compositionally appropriate alternative:

$$d_A(x, y) = \| \text{clr}(x) - \text{clr}(y) \|_2, \quad \text{clr}(x)_j = \log\frac{x_j}{g(x)}$$

Zeros handled by **multiplicative replacement** (preserves ratios among non-zero parts).

In [ ]:
def multiplicative_replacement(mat: np.ndarray, delta: float | None = None) -> np.ndarray:
    """Replace zeros in a composition matrix while preserving row sums = 1."""
    mat = np.asarray(mat, dtype=float)
    n_features = mat.shape[1]
    if delta is None:
        delta = 1.0 / n_features ** 2
    result = mat.copy()
    for i in range(len(result)):
        row = result[i]
        zero_mask = row == 0
        k = zero_mask.sum()
        if k == 0:
            continue
        result[i, zero_mask] = delta
        result[i, ~zero_mask] *= 1.0 - k * delta
    return result

# ── CLR transform ─────────────────────────────────────────────────────────────
rel = count_df.div(count_df.sum(axis=1), axis=0)
comp = multiplicative_replacement(rel.values)
clr_matrix = clr(comp)   # (n_samples, n_features)

# ── Aitchison distance = Euclidean on CLR ─────────────────────────────────────
aitchison_condensed = pdist(clr_matrix, metric="euclidean")
aitchison_matrix    = squareform(aitchison_condensed)
aitchison_df        = pd.DataFrame(aitchison_matrix, index=sample_ids, columns=sample_ids)
aitchison_linkage   = linkage(aitchison_condensed, method="average")

# ── BC distance for comparison ────────────────────────────────────────────────
from gamma_diversity.core._baseline_utils import bray_curtis_dissimilarity
bc_matrix = bray_curtis_dissimilarity(rel_abund_df.values)
bc_df     = pd.DataFrame(bc_matrix, index=sample_ids, columns=sample_ids)
bc_linkage = linkage(squareform(bc_matrix), method="average")

triu_ait = aitchison_matrix[np.triu_indices_from(aitchison_matrix, k=1)]
triu_bc  = bc_matrix[np.triu_indices_from(bc_matrix, k=1)]

print(f"Aitchison: mean={triu_ait.mean():.3f}  median={np.median(triu_ait):.3f}  range=[{triu_ait.min():.2f}, {triu_ait.max():.2f}]")
print(f"BC:        mean={triu_bc.mean():.3f}  median={np.median(triu_bc):.3f}  range=[{triu_bc.min():.4f}, {triu_bc.max():.4f}]")

# ── Side-by-side histograms ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, vals, label, color in [
    (axes[0], triu_bc,  "Bray-Curtis",     "#DD8452"),
    (axes[1], triu_ait, "Aitchison (CLR)", "#4C72B0"),
]:
    ax.hist(vals, bins=60, color=color, alpha=0.8, edgecolor="white", linewidth=0.3)
    ax.axvline(vals.mean(),     color="black", linestyle="--", linewidth=1.2,
               label=f"mean = {vals.mean():.3f}")
    ax.axvline(np.median(vals), color="grey",  linestyle=":",  linewidth=1.2,
               label=f"median = {np.median(vals):.3f}")
    ax.set_xlabel(f"{label} distance")
    ax.set_ylabel("pair count")
    ax.set_title(f"Pairwise {label}\n({len(sample_ids)} samples)")
    ax.legend(fontsize=8)
plt.suptitle("Distribution of pairwise distances (all sample pairs)", y=1.02, fontsize=11)
plt.tight_layout()
plt.show()

## CLR-transformed abundance heatmap

Same top-50 ASVs, but values are **CLR-transformed**.  
The colour scale is symmetric around zero: **red** = above geometric mean, **blue** = below.  
Row order follows Aitchison linkage (compositionally coherent); ASV columns clustered fresh on CLR values.

In [ ]:
clr_df    = pd.DataFrame(clr_matrix, index=sample_ids, columns=asv_ids)
clr_top50 = clr_df[top_asvs].rename(columns=final_rename)

row_colors_clr = row_colors_samples.loc[clr_top50.index]

g_clr = sns.clustermap(
    clr_top50,
    row_linkage=aitchison_linkage,
    col_cluster=True,
    method="average",
    metric="euclidean",
    row_colors=row_colors_clr,
    figsize=(20, 24),
    yticklabels=False,
    xticklabels=True,
    cmap="RdBu_r",
    center=0,
    cbar_pos=None,
    dendrogram_ratio=(0.10, 0.15),
    colors_ratio=0.010,
)
g_clr.ax_heatmap.set_xticklabels(
    g_clr.ax_heatmap.get_xticklabels(), rotation=90, fontsize=9
)
g_clr.fig.subplots_adjust(top=0.72)

# Colorbar
clr_vals = clr_top50.values
cax_clr  = g_clr.fig.add_axes([0.02, 0.755, 0.18, 0.006])
vmax_clr = max(abs(clr_vals.min()), abs(clr_vals.max()))
norm_clr = mpl.colors.TwoSlopeNorm(vmin=-vmax_clr, vcenter=0, vmax=vmax_clr)
cb_clr   = g_clr.fig.colorbar(mpl.cm.ScalarMappable(norm=norm_clr, cmap="RdBu_r"),
                               cax=cax_clr, orientation="horizontal")
cb_clr.set_label("CLR value", fontsize=10)
cb_clr.ax.tick_params(labelsize=9)

g_clr.fig.suptitle(
    f"CLR-transformed abundance — {len(sample_ids)} samples × top-{N_TOP} ASVs\n"
    f"(bacteria only, decontaminated; {n_asvs_total} ASVs total) "
    "— Aitchison row order, average-linkage ASV columns",
    x=0.5, y=0.995, ha="center", fontsize=13,
)
add_legends(g_clr.fig)
plt.show()

## Bray-Curtis dissimilarity matrix

Pairwise BC dissimilarity for all samples (controls included).  
Precomputed linkage (average) applied to both row and column dendrograms.

In [ ]:
row_colors_bc = pd.DataFrame({
    "sample type": [SAMPLE_TYPE_COLORS[sample_to_group[s]] for s in sample_ids],
    "farm":        [farm_color(s) for s in sample_ids],
    "tree":        [tree_color(s) for s in sample_ids],
}, index=sample_ids)

g_bc = sns.clustermap(
    bc_df,
    row_linkage=bc_linkage,
    col_linkage=bc_linkage,
    row_colors=row_colors_bc,
    col_colors=row_colors_bc,
    figsize=(18, 18),
    yticklabels=False,
    xticklabels=False,
    cmap="YlOrRd",
    cbar_pos=None,
    dendrogram_ratio=(0.10, 0.10),
    colors_ratio=0.012,
    vmin=0, vmax=1,
)
g_bc.fig.subplots_adjust(top=0.78)

# Colorbar
cax_bc  = g_bc.fig.add_axes([0.02, 0.81, 0.18, 0.006])
norm_bc = mpl.colors.Normalize(vmin=0, vmax=1)
cb_bc   = g_bc.fig.colorbar(mpl.cm.ScalarMappable(norm=norm_bc, cmap="YlOrRd"),
                             cax=cax_bc, orientation="horizontal")
cb_bc.set_label("Bray-Curtis dissimilarity", fontsize=10)
cb_bc.ax.tick_params(labelsize=9)

g_bc.fig.suptitle(
    f"Bray-Curtis dissimilarity matrix — {len(sample_ids)} samples (16S)\n"
    "Average linkage on precomputed distances",
    x=0.5, y=0.995, ha="center", fontsize=13,
)
add_legends(g_bc.fig, y_stype=0.960, y_farm=0.930, y_tree=0.895)
plt.show()

## Aitchison distance matrix

Euclidean distance on CLR-transformed compositions (same CLR computed above).  
Zeros replaced by multiplicative replacement before CLR.

In [ ]:
row_colors_ait = row_colors_bc.copy()   # same annotation strips

g_ait = sns.clustermap(
    aitchison_df,
    row_linkage=aitchison_linkage,
    col_linkage=aitchison_linkage,
    row_colors=row_colors_ait,
    col_colors=row_colors_ait,
    figsize=(18, 18),
    yticklabels=False,
    xticklabels=False,
    cmap="YlOrRd",
    cbar_pos=None,
    dendrogram_ratio=(0.10, 0.10),
    colors_ratio=0.012,
)
g_ait.fig.subplots_adjust(top=0.78)

# Colorbar
cax_ait  = g_ait.fig.add_axes([0.02, 0.81, 0.18, 0.006])
norm_ait = mpl.colors.Normalize(vmin=triu_ait.min(), vmax=triu_ait.max())
cb_ait   = g_ait.fig.colorbar(mpl.cm.ScalarMappable(norm=norm_ait, cmap="YlOrRd"),
                               cax=cax_ait, orientation="horizontal")
cb_ait.set_label("Aitchison distance (Euclidean on CLR)", fontsize=10)
cb_ait.ax.tick_params(labelsize=9)

g_ait.fig.suptitle(
    f"Aitchison distance matrix — {len(sample_ids)} samples (16S, bacteria, decontaminated)\n"
    "Multiplicative replacement → CLR → Euclidean, average linkage",
    x=0.5, y=0.995, ha="center", fontsize=13,
)
add_legends(g_ait.fig, y_stype=0.960, y_farm=0.930, y_tree=0.895)
plt.show()